# Full LLM Project Activity Pipeline (TF-IDF + Logistic Regression + Pre-trained Model Inference)

In [ ]:
# ✅ Step 1: Install Dependencies
!pip install --quiet numpy==1.26.4 pandas==1.5.3 scipy==1.10.1
!pip install --quiet gensim==4.3.2 pyldavis==3.4.1 datasets==2.18.0 nltk scikit-learn transformers evaluate accelerate huggingface_hub
!apt-get install -y git-lfs

In [ ]:
# ✅ Step 2: Import Libraries
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import evaluate
from huggingface_hub import notebook_login
import os

In [ ]:
# ✅ Step 3: Download NLTK Resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
# ✅ Step 4: Define Preprocessing and Pipeline Functions
top_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    tokens = text.split()
    return ' '.join([lemmatizer.lemmatize(word) for word in tokens if word not in top_words])

def load_and_preprocess():
    dataset = load_dataset("SetFit/20_newsgroups")
    df = pd.DataFrame(dataset['train'])
    df['clean_text'] = df['text'].apply(preprocess_text)
    return df

def vectorize_text(df, method='tfidf'):
    if method == 'tfidf':
        vectorizer = TfidfVectorizer()
    elif method == 'bow':
        vectorizer = CountVectorizer()
    else:
        raise ValueError("Method must be 'tfidf' or 'bow'")
    X = vectorizer.fit_transform(df['clean_text'])
    y = df['label']
    print(f"Number of documents: {X.shape[0]}")
    print(f"Vocabulary size: {X.shape[1]}")
    return X, y, vectorizer

def train_evaluate_save_model(X, y, model_path):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print("\n📊 Classification Report:\n")
    print(classification_report(y_test, y_pred))
    print("✅ Accuracy:", accuracy_score(y_test, y_pred))
    joblib.dump(clf, model_path)
    print(f"✅ Model saved to {model_path}")

In [ ]:
# ✅ Step 5: Execute Classic ML Pipeline
df = load_and_preprocess()
X_tfidf, y_tfidf, tfidf_vectorizer = vectorize_text(df, method='tfidf')
train_evaluate_save_model(X_tfidf, y_tfidf, "tfidf_logreg_model.pkl")
X_bow, y_bow, bow_vectorizer = vectorize_text(df, method='bow')
train_evaluate_save_model(X_bow, y_bow, "bow_logreg_model.pkl")

In [ ]:
# ✅ Step 6: Inference with Public Emotion Model for Demo
print("\n🤖 Running inference with a public LLM (emotion classification)...")
sample_texts = df['text'].iloc[:5].tolist()
classifier = pipeline(task="text-classification", model="bhadresh-savani/distilbert-base-uncased-emotion")
preds = classifier(sample_texts)
for text, pred in zip(sample_texts, preds):
    print(f"Text: {text[:100]}...")
    print(f"Prediction: {pred['label']} (Confidence: {pred['score']:.4f})\n")

In [ ]:
# ✅ Step 7: Fine-tuning a Transformer Model on 20 Newsgroups
from datasets import ClassLabel
raw_datasets = load_dataset("SetFit/20_newsgroups")
label_features = ClassLabel(num_classes=20)
raw_datasets = raw_datasets.rename_column("label", "labels")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=20)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

tokenized_ds = raw_datasets.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])

def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), axis=1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
# ✅ Step 8: Optimize and Push Model to Hugging Face Hub
notebook_login()
repo_name = "finetuned-newsgroups-distilbert"

training_args = TrainingArguments(
    output_dir=repo_name,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=10,
    push_to_hub=True,
)

os.environ["WANDB_DISABLED"] = "true"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()
trainer.push_to_hub()

print("✅ Model pushed to Hugging Face Hub. You can now view and use it from your account.")